In [1]:
import numpy as np
import pandas as pd
import scarf

scarf.fetch_dataset(
    dataset_name="kang_15K_pbmc_rnaseq", save_path="scarf_datasets", as_zarr=True
)
scarf.fetch_dataset(
    dataset_name="kang_14K_ifnb-pbmc_rnaseq", save_path="scarf_datasets", as_zarr=True
)

ds_ctrl = scarf.DataStore("scarf_datasets/kang_15K_pbmc_rnaseq/data.zarr")
ds_stim = scarf.DataStore("scarf_datasets/kang_14K_ifnb-pbmc_rnaseq/data.zarr")

In [2]:
ds_ctrl.cells.insert(
    "reference_batch", np.repeat("control", ds_ctrl.cells.N), overwrite=True
)

reference = ds_ctrl.build_mapping_reference(
    feat_key="hvgs", batch_columns=["reference_batch"]
)

make_graph step 1/7: normalize expression matrix (reusing cached)



Using existing normalized data with cell key I and feat key I__hvgs



make_graph step 1/7: normalize expression matrix finished in 0.2s (reusing cached)



make_graph step 2/7: normalization statistics (computing)



make_graph step 2/7: normalization statistics finished in 0.5s (computing)



...

make_graph step 6/7: smooth KNN distances into graph finished in 16.9s (computing)



ANN recall: 99.92%



make_graph step 7/7: finalize graph metadata (computing)



make_graph step 7/7: finalize graph metadata finished in 0.0s (computing)



make_graph finished in 27.4s (7/7 steps, 26.5s in logged steps)



In [3]:
reference = ds_ctrl.get_mapping_reference(feat_key="hvgs")

In [4]:
result = reference.map_query(
    target_assay=ds_stim.RNA,
    target_name="stim_symphony",
    target_feat_key="hvgs_symphony",
    save_k=5,
    query_batches=pd.DataFrame(
        {
            "reference_batch": np.repeat(
                "stimulated", len(ds_stim.cells.fetch("ids", key="I"))
            )
        }
    ),
)
result

0 features missing in target data



Loaded existing ANN stream from RNA/normed__I__hvgs/reduction__pca__25__I/ann__l2__50__50__48__4466__harmony_831a799898edc86d



MappingResult(projection_path='RNA/projections/stim_symphony', n_cells=10111, correction_method='symphony', diagnostics={'featureCoverage': 1.0, 'queryBatchCount': 1.0, 'zeroNormCellCount': 0.0, 'algorithmVariant': 'symphonyStyleV1'}, indices=None, distances=None, uncorrected_latent=None, corrected_latent=None, uninformative=None)

In [5]:
projection = ds_ctrl.z["RNA"]["projections"]["stim_symphony"]
dict(projection.attrs)

uncorrected = projection["uncorrectedLatent"][:]
corrected = projection["correctedLatent"][:]
np.mean(np.abs(corrected - uncorrected))

np.float64(1.5035957638686113)

In [6]:
evidence = ds_ctrl.get_target_label_evidence(
    target_name="stim_symphony",
    reference_class_group="cluster_labels",
    threshold_fraction=0.6,
)
evidence.head()

,label,voteFraction,voteEntropy,topTwoMargin,featureCoverage,referenceDistancePercentile,isUnknown
0,CD8 T,0.619149,0.664479,0.238298,1.0,0.718168,False
1,pDC,1.000000,-0.000000,1.000000,1.0,0.856477,False
2,CD4 naive T,1.000000,-0.000000,1.000000,1.0,0.695549,False
3,B,1.000000,-0.000000,1.000000,1.0,0.863320,False
4,CD4 naive T,0.792535,0.510579,0.585070,1.0,0.503071,False


In [7]:
calibrated = ds_ctrl.get_target_label_evidence(
    target_name="stim_symphony",
    reference_class_group="cluster_labels",
    calibration_nonconformity=np.array([0.08, 0.12, 0.2, 0.25]),
    conformal_alpha=0.1,
)
calibrated[["label", "predictionSet"]].head()

,label,predictionSet
0,CD8 T,"(CD4 naive T, CD 14 Mono, pDC, CD4 Memory T, T..."
1,pDC,"(CD4 naive T, CD 14 Mono, pDC, CD4 Memory T, T..."
2,CD4 naive T,"(CD4 naive T, CD 14 Mono, pDC, CD4 Memory T, T..."
3,B,"(CD4 naive T, CD 14 Mono, pDC, CD4 Memory T, T..."
4,CD4 naive T,"(CD4 naive T, CD 14 Mono, pDC, CD4 Memory T, T..."


In [8]:
control_result = reference.map_query(
    target_assay=ds_ctrl.RNA,
    target_name="control_symphony",
    target_feat_key="hvgs_control_symphony",
    save_k=5,
    query_batches=pd.DataFrame(
        {"reference_batch": ds_ctrl.cells.fetch("reference_batch", key="I")}
    ),
)
control_result

0 features missing in target data



Loaded existing ANN stream from RNA/normed__I__hvgs/reduction__pca__25__I/ann__l2__50__50__48__4466__harmony_831a799898edc86d



MappingResult(projection_path='RNA/projections/control_symphony', n_cells=8487, correction_method='symphony', diagnostics={'featureCoverage': 1.0, 'queryBatchCount': 1.0, 'zeroNormCellCount': 0.0, 'algorithmVariant': 'symphonyStyleV1'}, indices=None, distances=None, uncorrected_latent=None, corrected_latent=None, uninformative=None)